In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", device_map="auto", torch_dtype=torch.bfloat16)

messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate",},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
print(tokenizer.decode(tokenized_chat[0]))

/home/guilherme/miniconda3/envs/multistep/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


<｜begin▁of▁sentence｜>You are a friendly chatbot who always responds in the style of a pirate<｜User｜>How many helicopters can a human eat in one sitting?<｜Assistant｜><think>



In [2]:
import numpy as np

tokenized_chat_lst = torch.as_tensor(np.array([tokenized_chat, tokenized_chat])).to('cuda')

In [3]:
outputs = model.generate(tokenized_chat_lst[0], max_new_tokens=1024) 
print(tokenizer.decode(outputs[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<｜begin▁of▁sentence｜>You are a friendly chatbot who always responds in the style of a pirate<｜User｜>How many helicopters can a human eat in one sitting?<｜Assistant｜><think>
Alright, so I'm trying to figure out how many helicopters a human can eat in one sitting. Hmm, that's an interesting question. First off, I know that helicopters are expensive and they're used for various purposes, like taking off, landing, and even for cargo. But can a human even think about eating them?

Maybe the question is more about comparing the costs or the resources needed to operate a helicopter versus a human. Or perhaps it's a playful way to ask how much effort or resources a human would need to handle something as big and expensive as a helicopter.

I should consider the context in which the question is asked. If it's a real-world scenario, maybe it's about comparing the cost of operating a helicopter versus a human, but that doesn't make much sense because humans aren't capable of handling such large r

In [4]:
def extract_after_last_assistant(text: str) -> str:
    marker = "<｜Assistant｜>"
    eos_marker = "<｜end▁of▁sentence｜>"
    last_occurrence = text.rfind(marker)
    if last_occurrence == -1:
        return ""  # Return empty string if marker is not found
    result = text[last_occurrence + len(marker):].strip()
    if result.endswith(eos_marker):
        result = result[:-len(eos_marker)].strip()
    return result

message_ = extract_after_last_assistant(tokenizer.decode(outputs[0]))
messages.append({"role":"assistant", "content": message_})

In [5]:
messages

[{'role': 'system',
  'content': 'You are a friendly chatbot who always responds in the style of a pirate'},
 {'role': 'user',
  'content': 'How many helicopters can a human eat in one sitting?'},
 {'role': 'assistant',
  'content': "<think>\nAlright, so I'm trying to figure out how many helicopters a human can eat in one sitting. Hmm, that's an interesting question. First off, I know that helicopters are expensive and they're used for various purposes, like taking off, landing, and even for cargo. But can a human even think about eating them?\n\nMaybe the question is more about comparing the costs or the resources needed to operate a helicopter versus a human. Or perhaps it's a playful way to ask how much effort or resources a human would need to handle something as big and expensive as a helicopter.\n\nI should consider the context in which the question is asked. If it's a real-world scenario, maybe it's about comparing the cost of operating a helicopter versus a human, but that does

In [6]:
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
print(tokenizer.decode(tokenized_chat[0]))

<｜begin▁of▁sentence｜>You are a friendly chatbot who always responds in the style of a pirate<｜User｜>How many helicopters can a human eat in one sitting?<｜Assistant｜>

The question posed is more of a metaphor or joke than a literal one. It's a playful way to explore the costs or resources associated with operating a helicopter versus a human, but in reality, it's not feasible to compare these two due to their vastly different capabilities and purposes. Therefore, the answer is that the question is more of a metaphor or joke, and the intent likely lies elsewhere.<｜end▁of▁sentence｜><｜Assistant｜><think>

